# REINVENT4 + Case v2.1 — полный анализ и визуализация результатов (с пояснениями)

Этот notebook предназначен для архива **`REINVENT4_MOST_PARETO_RESULTS.zip`** и объединяет весь анализ в одном месте.

В успешном эксперименте получено:

- **7** независимых Pareto-профилей;
- **3480** строк финального scoring по профилям;
- **1307** уникальных молекулярных кандидатов;
- **823** кандидата прошли одновременно AD + SAScore;
- **319** кандидатов вошли в глобальный Pareto-front;
- **275** кандидатов прошли strict-фильтр;
- **3** кандидата прошли robust-фильтр.

Notebook строит:

1. автоматическую распаковку и проверку архива;
2. воронку отбора кандидатов;
3. сравнение 7 обученных агентов;
4. pass-rate по каждому из 7 свойств;
5. распределения прогнозов и пороги;
6. корреляции свойств и Pareto utilities;
7. анализ глобального Pareto-front;
8. объяснение перехода `275 strict → 3 robust`;
9. PCA химического пространства по Morgan fingerprints;
10. scaffold-анализ;
11. таблицы и 2D-структуры strict/robust-кандидатов;
12. экспорт таблиц, PNG/HTML-графиков и итогового ZIP с анализом.

> **Важно:** `deltaH_activation_proxy_kJ_mol` — это кинетический Eyring activation-barrier proxy при предположении `ΔS‡≈0`. Это **не** термодинамическая энергия хранения MOST `ΔH_storage`.

## Логика анализа: от генерации к научному выводу

Этот notebook **не обучает REINVENT заново**. Он берёт уже завершённый эксперимент и отвечает на вопрос: *что именно получилось и насколько результат выглядит убедительно внутри нашей computational постановки?*

Основные уровни кандидатов:

- **Unique** — все уникальные молекулы после объединения семи REINVENT-профилей.
- **AD + SAS** — кандидаты, для которых прогнозы находятся в более приемлемой области применимости и структура проходит synthetic-accessibility фильтр.
- **Global Pareto front** — недоминируемые компромиссы между семью utilities.
- **Strict** — центральные прогнозы одновременно удовлетворяют всем заданным порогам.
- **Robust** — strict-кандидаты, которые сохраняют запас относительно порогов при более консервативном учёте uncertainty.

### Что в этом notebook считается свидетельством хорошего результата

Мы смотрим не на одну цифру, а на совокупность признаков: validity, uniqueness/novelty, AD, success rates, Pareto structure, diversity/scaffolds, uncertainty и robust shortlist. Это защищает от ситуации, когда высокий reward получен ценой повторов, выхода за AD или эксплуатации ошибки surrogate-модели.

### Научное ограничение

Все свойства здесь — **модельные прогнозы**, а не лабораторные измерения. Поэтому `strict` и `robust` означают «проходит вычислительный отбор», а не «доказанно безопасен/эффективен». Особенно важно помнить, что `deltaH_activation_proxy_kJ_mol` — Eyring activation-barrier proxy, вычисленный из half-life, а не настоящая MOST storage enthalpy.


## 1. Установка библиотек

Ячейка проверяет зависимости и устанавливает только отсутствующие пакеты. В Google Colab обычно большая часть уже присутствует.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "plotly": "plotly",
    "rdkit": "rdkit",
}

missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Устанавливаю:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Все основные зависимости уже установлены.")

## 2. Импорт и настройки

In [ ]:
from pathlib import Path
import json
import math
import os
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
INPUT_DIR = BASE / "REINVENT4_MOST_PARETO_ANALYSIS_INPUT"
OUT_DIR = BASE / "REINVENT4_MOST_PARETO_ANALYSIS"
FIG_DIR = OUT_DIR / "figures"
HTML_DIR = OUT_DIR / "interactive"
TABLE_DIR = OUT_DIR / "tables"

for d in [OUT_DIR, FIG_DIR, HTML_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Рабочая папка:", BASE)
print("Результаты анализа будут сохранены в:", OUT_DIR)

## 3. Найти или загрузить `REINVENT4_MOST_PARETO_RESULTS.zip`

Сначала notebook ищет архив в `/content` и Google Drive. Если файл не найден, в Colab откроется обычное окно загрузки.

In [ ]:
def locate_results_zip():
    candidates = []
    exact_locations = [
        BASE / "REINVENT4_MOST_PARETO_RESULTS.zip",
        BASE / "drive" / "MyDrive" / "REINVENT4_MOST_PARETO_RESULTS.zip",
    ]
    for p in exact_locations:
        if p.is_file():
            candidates.append(p)

    if not candidates:
        for root in [BASE, BASE / "drive" / "MyDrive"]:
            if root.exists():
                candidates.extend(root.rglob("REINVENT4_MOST_PARETO_RESULTS.zip"))

    if candidates:
        candidates = sorted(set(candidates), key=lambda p: (len(str(p)), str(p)))
        return candidates[0]

    try:
        from google.colab import files
        print("Архив не найден автоматически. Выберите REINVENT4_MOST_PARETO_RESULTS.zip")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("Файл не был загружен")
        names = list(uploaded)
        preferred = [n for n in names if n.endswith("REINVENT4_MOST_PARETO_RESULTS.zip")]
        name = preferred[0] if preferred else names[0]
        return BASE / name
    except ImportError as e:
        raise FileNotFoundError(
            "REINVENT4_MOST_PARETO_RESULTS.zip не найден. Положите его рядом с notebook."
        ) from e

RESULTS_ZIP = locate_results_zip()
assert RESULTS_ZIP.is_file(), RESULTS_ZIP
print("Найден архив:", RESULTS_ZIP)
print("Размер, MB:", round(RESULTS_ZIP.stat().st_size / 1024**2, 2))

with zipfile.ZipFile(RESULTS_ZIP, "r") as z:
    bad = z.testzip()
    print("Проверка ZIP:", "OK" if bad is None else f"ПОВРЕЖДЁН: {bad}")
    print("Файлов в ZIP:", len(z.namelist()))

## 4. Безопасная распаковка и поиск агрегированных таблиц

In [ ]:
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_extract(zip_path: Path, destination: Path):
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as z:
        for info in z.infolist():
            target = (destination / info.filename).resolve()
            if root != target and root not in target.parents:
                raise RuntimeError(f"Небезопасный путь в ZIP: {info.filename}")
        z.extractall(destination)


safe_extract(RESULTS_ZIP, INPUT_DIR)
print("Распаковано в:", INPUT_DIR)


def find_result(name: str, required=True):
    matches = list(INPUT_DIR.rglob(name))
    # Файлы из aggregate имеют приоритет.
    matches.sort(key=lambda p: (0 if "aggregate" in p.parts else 1, len(str(p))))
    if not matches:
        if required:
            raise FileNotFoundError(f"Не найден {name} внутри {RESULTS_ZIP.name}")
        return None
    return matches[0]

paths = {
    "summary": find_result("summary.json"),
    "profile_summary": find_result("profile_summary.csv"),
    "all_rows": find_result("all_profile_rows.csv"),
    "all_unique": find_result("all_unique_candidates.csv"),
    "pareto": find_result("global_pareto_front.csv"),
    "strict": find_result("strict_candidates.csv"),
    "robust": find_result("robust_candidates.csv"),
    "manifest": find_result("experiment_manifest.json", required=False),
}

for k, p in paths.items():
    print(f"{k:16s}", p)

## 5. Загрузка результатов

In [ ]:
with open(paths["summary"], "r", encoding="utf-8") as f:
    summary = json.load(f)

profile_summary = pd.read_csv(paths["profile_summary"])
all_rows = pd.read_csv(paths["all_rows"])
all_unique = pd.read_csv(paths["all_unique"])
pareto = pd.read_csv(paths["pareto"])
strict = pd.read_csv(paths["strict"])
robust = pd.read_csv(paths["robust"])

manifest = None
if paths["manifest"] is not None:
    with open(paths["manifest"], "r", encoding="utf-8") as f:
        manifest = json.load(f)


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
        .astype(bool)
    )

bool_candidates = [
    "valid", "novelty", "inside_both_ad", "sascore_pass",
    "joint_pass", "robust_joint_pass",
]
for df in [all_rows, all_unique, pareto, strict, robust]:
    for c in df.columns:
        if c in bool_candidates or c.startswith("pass_") or c.startswith("robust_pass_"):
            df[c] = as_bool(df[c])

print("summary.json")
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\nРазмеры таблиц:")
for name, df in [
    ("all_profile_rows", all_rows),
    ("all_unique_candidates", all_unique),
    ("global_pareto_front", pareto),
    ("strict_candidates", strict),
    ("robust_candidates", robust),
]:
    print(f"{name:26s}: {df.shape[0]:5d} строк × {df.shape[1]} колонок")

## 6. Проверка целостности результатов

Количество строк в ключевых таблицах должно совпадать с `summary.json`.
**Почему это важно.** До построения красивых графиков notebook проверяет, что размеры CSV согласуются с `summary.json`. Это простая защита от анализа неполного или случайно смешанного архива.


In [ ]:
checks = pd.DataFrame([
    ["Всего строк по профилям", len(all_rows), summary.get("profile_rows_total")],
    ["Уникальные кандидаты", len(all_unique), summary.get("unique_candidates")],
    ["Global Pareto front", len(pareto), summary.get("global_pareto_front")],
    ["Strict candidates", len(strict), summary.get("strict_candidates")],
    ["Robust candidates", len(robust), summary.get("robust_candidates")],
], columns=["Показатель", "Фактически", "В summary.json"])
checks["Совпадает"] = checks["Фактически"] == checks["В summary.json"]
display(checks)

if not checks["Совпадает"].all():
    print("ВНИМАНИЕ: есть расхождения между CSV и summary.json")
else:
    print("Проверка пройдена: ключевые количества согласованы.")

# Часть A. Общая статистика

## 7. Воронка отбора

Ключевой итог эксперимента: **1307 уникальных → 823 AD+SAS → 319 Pareto → 275 strict → 3 robust**.
**Как читать график.** Падение числа кандидатов на каждом уровне не означает «ошибку генератора». Чем дальше вправо/вниз по воронке, тем более строгий вопрос мы задаём к одной и той же структуре. Особенно резкое сужение `strict → robust` показывает, что многие кандидаты проходят порог близко к границе и чувствительны к неопределённости evaluator-а.


In [ ]:
funnel = pd.DataFrame({
    "Этап": ["Unique", "AD + SAS", "Pareto front", "Strict", "Robust"],
    "Кандидаты": [
        int(summary["unique_candidates"]),
        int(summary["inside_both_ad_and_sas"]),
        int(summary["global_pareto_front"]),
        int(summary["strict_candidates"]),
        int(summary["robust_candidates"]),
    ]
})
funnel["% от unique"] = 100 * funnel["Кандидаты"] / funnel.loc[0, "Кандидаты"]
display(funnel.round(2))
funnel.to_csv(TABLE_DIR / "selection_funnel.csv", index=False)

plt.figure(figsize=(9, 5))
plt.bar(funnel["Этап"], funnel["Кандидаты"])
plt.ylabel("Количество кандидатов")
plt.title("Воронка отбора REINVENT4 + Pareto")
for i, v in enumerate(funnel["Кандидаты"]):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_selection_funnel.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Параметры проведённого эксперимента

In [ ]:
if manifest:
    exp_meta = pd.DataFrame({"Параметр": list(manifest.keys()), "Значение": list(manifest.values())})
    display(exp_meta)
else:
    print("experiment_manifest.json не найден — это не мешает анализу CSV.")

## 9. Сравнение семи Pareto-профилей

`joint_pass` — количество кандидатов, прошедших все raw-пороги + AD + SAS в конкретном профиле.  
`robust_joint_pass` — более строгий вариант с учётом `0.5σ` неопределённости.
**Интерпретация.** Это сравнение семи одинаково устроенных REINVENT-агентов с разными priorities. Более высокий `joint_pass` не делает профиль «лучшей нейросетью» вообще: профили специально исследуют разные направления компромисса. Полезно смотреть, какие priorities чаще приводят к совместному прохождению требований и какие дают уникальные robust-кандидаты.


In [ ]:
ps = profile_summary.copy()
ps = ps.sort_values("joint_pass", ascending=True)
display(ps)
profile_summary.to_csv(TABLE_DIR / "profile_summary.csv", index=False)

plt.figure(figsize=(10, 6))
y = np.arange(len(ps))
h = 0.35
plt.barh(y - h/2, ps["joint_pass"], height=h, label="Strict / joint_pass")
plt.barh(y + h/2, ps["robust_joint_pass"], height=h, label="Robust")
plt.yticks(y, ps["priority"])
plt.xlabel("Количество кандидатов")
plt.title("Strict и robust результаты по 7 профилям")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "02_profiles_pass_counts.png", dpi=180, bbox_inches="tight")
plt.show()

ps_reward = profile_summary.sort_values("mean_reward", ascending=True)
plt.figure(figsize=(10, 5))
plt.barh(ps_reward["priority"], ps_reward["mean_reward"])
plt.xlabel("Mean reward")
plt.title("Средний reward по Pareto-профилям")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_profiles_mean_reward.png", dpi=180, bbox_inches="tight")
plt.show()

# Часть B. Семь целевых свойств

Использованные ограничения Case v2.1:

- `absorption_max_nm`: 290–420 nm;
- `log_extinction`: ≥ 3.80;
- `photochem_efficiency`: ≥ 0.25;
- `log_half_life`: ≥ 3.56 log10(s), то есть примерно ≥ 1 часа;
- `log_kp`: ≤ -5.00;
- `skin_sensitization`: ≤ 0.50;
- `skin_irritation`: ≤ 0.50.

Дополнительно: AD distance ≤ 0.70 в обеих группах и SAScore ≤ 5.0.

In [ ]:
PROPS = {
    "group_A_absorption_max_nm": {
        "label": "Absorption max, nm", "kind": "window", "low": 290.0, "high": 420.0, "scale": 44.86034262567687
    },
    "group_A_log_extinction": {
        "label": "log extinction", "kind": "minimum", "threshold": 3.80, "scale": 0.3374392036949682
    },
    "group_A_photochem_efficiency": {
        "label": "Photochem efficiency", "kind": "minimum", "threshold": 0.25, "scale": 0.22730020885430102
    },
    "group_A_log_half_life": {
        "label": "log10(t1/2, s)", "kind": "minimum", "threshold": 3.56, "scale": 0.7143776834370066
    },
    "group_B_log_kp": {
        "label": "log Kp", "kind": "maximum", "threshold": -5.00, "scale": 0.7069483885565052
    },
    "group_B_skin_sensitization": {
        "label": "Skin sensitization probability", "kind": "maximum", "threshold": 0.50, "scale": 0.27192429147762764
    },
    "group_B_skin_irritation": {
        "label": "Skin irritation probability", "kind": "maximum", "threshold": 0.50, "scale": 0.2613740870108099
    },
}

AD_MAX = 0.70
SAS_MAX = 5.0
UNCERTAINTY_BETA = 0.5

prop_table = []
for k, cfg in PROPS.items():
    if cfg["kind"] == "window":
        rule = f'{cfg["low"]} ≤ x ≤ {cfg["high"]}'
    elif cfg["kind"] == "minimum":
        rule = f'x ≥ {cfg["threshold"]}'
    else:
        rule = f'x ≤ {cfg["threshold"]}'
    prop_table.append([k, cfg["label"], rule, cfg["scale"]])

display(pd.DataFrame(prop_table, columns=["property", "Название", "Strict rule", "Utility scale"]))

## 10. Pass-rate каждого свойства среди кандидатов внутри AD + SAS

Это показывает, какой критерий чаще всего является ограничивающим.
**Зачем это нужно.** Если почти все кандидаты проходят один порог, но часто проваливают другой, второй критерий является реальным bottleneck поиска. Это помогает понять, где ограничение химическое, а где, возможно, связано с неопределённостью/слабостью evaluator-а.


In [ ]:
feasible_mask = as_bool(all_unique["inside_both_ad"]) & as_bool(all_unique["sascore_pass"])
feasible = all_unique.loc[feasible_mask].copy()

pass_rows = []
for key, cfg in PROPS.items():
    raw_col = f"pass_{key}"
    robust_col = f"robust_pass_{key}"
    raw_rate = 100 * as_bool(feasible[raw_col]).mean()
    robust_rate = 100 * as_bool(feasible[robust_col]).mean()
    pass_rows.append([key, cfg["label"], raw_rate, robust_rate, raw_rate - robust_rate])

pass_rates = pd.DataFrame(pass_rows, columns=[
    "property", "label", "strict_pass_%", "robust_pass_%", "drop_pp"
]).sort_values("strict_pass_%")
display(pass_rates.round(2))
pass_rates.to_csv(TABLE_DIR / "property_pass_rates.csv", index=False)

x = np.arange(len(pass_rates))
w = 0.38
plt.figure(figsize=(11, 6))
plt.bar(x - w/2, pass_rates["strict_pass_%"], width=w, label="Raw / strict")
plt.bar(x + w/2, pass_rates["robust_pass_%"], width=w, label="Robust 0.5σ")
plt.xticks(x, pass_rates["label"], rotation=35, ha="right")
plt.ylabel("Pass rate среди AD+SAS, %")
plt.title("Какие свойства сильнее ограничивают отбор")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "04_property_pass_rates.png", dpi=180, bbox_inches="tight")
plt.show()

## 11. Распределения предсказанных свойств

Для каждого свойства показываются кандидаты внутри AD+SAS и отдельно global Pareto-front. Вертикальные линии — strict-пороги.
**Что искать на графиках.** Нас интересует не только среднее значение, но и положение распределения относительно порога. Хороший признак — когда Pareto-кандидаты систематически сдвинуты в допустимую область, а не просто имеют несколько экстремальных выбросов.


In [ ]:
for key, cfg in PROPS.items():
    col = f"pred_{key}"
    if col not in all_unique.columns:
        print("Нет колонки:", col)
        continue

    a = pd.to_numeric(feasible[col], errors="coerce").dropna()
    b = pd.to_numeric(pareto[col], errors="coerce").dropna()

    plt.figure(figsize=(8.5, 4.8))
    plt.hist(a, bins=35, alpha=0.55, label=f"AD+SAS (n={len(a)})")
    plt.hist(b, bins=35, alpha=0.55, label=f"Pareto (n={len(b)})")

    if cfg["kind"] == "window":
        plt.axvline(cfg["low"], linestyle="--", linewidth=1.5)
        plt.axvline(cfg["high"], linestyle="--", linewidth=1.5)
    else:
        plt.axvline(cfg["threshold"], linestyle="--", linewidth=1.5)

    plt.xlabel(cfg["label"])
    plt.ylabel("Количество")
    plt.title(cfg["label"])
    plt.legend()
    plt.tight_layout()
    safe = key.replace("group_", "")
    plt.savefig(FIG_DIR / f"dist_{safe}.png", dpi=180, bbox_inches="tight")
    plt.show()

## 12. Корреляция семи предсказанных свойств

Используется Spearman correlation, чтобы не требовать линейной зависимости.
**Почему Spearman.** Она показывает монотонные связи и не требует линейности. Отрицательная корреляция между полезными направлениями указывает на настоящий trade-off: улучшение одного свойства статистически сопровождается ухудшением другого. Именно для таких случаев Pareto-подход особенно полезен.


In [ ]:
pred_cols = [f"pred_{k}" for k in PROPS]
pred_labels = [PROPS[k]["label"] for k in PROPS]

corr = all_unique[pred_cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
corr.index = pred_labels
corr.columns = pred_labels
display(corr.round(2))
corr.to_csv(TABLE_DIR / "prediction_spearman_correlation.csv")

plt.figure(figsize=(9, 8))
im = plt.imshow(corr.values, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, label="Spearman ρ")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Корреляции предсказанных свойств")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_prediction_correlation.png", dpi=180, bbox_inches="tight")
plt.show()

## 13. Корреляция Pareto utilities

Все семь objectives в Pareto-сортировке ориентированы одинаково: **больше utility = лучше**. Это удобнее для анализа trade-off, чем raw predictions с разным направлением критериев.
**Важно.** В raw-прогнозах некоторые свойства нужно уменьшать (`logKp`, probabilities), а другие увеличивать. Utilities уже приведены к единому направлению «больше = лучше», поэтому их корреляционная матрица удобнее для анализа конфликтов непосредственно внутри Pareto-задачи.


In [ ]:
utility_cols = [f"utility_{k}" for k in PROPS]
utility_labels = [PROPS[k]["label"] for k in PROPS]

ucorr = feasible[utility_cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
ucorr.index = utility_labels
ucorr.columns = utility_labels
display(ucorr.round(2))
ucorr.to_csv(TABLE_DIR / "utility_spearman_correlation.csv")

plt.figure(figsize=(9, 8))
im = plt.imshow(ucorr.values, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, label="Spearman ρ")
plt.xticks(range(len(ucorr.columns)), ucorr.columns, rotation=45, ha="right")
plt.yticks(range(len(ucorr.index)), ucorr.index)
for i in range(len(ucorr.index)):
    for j in range(len(ucorr.columns)):
        plt.text(j, i, f"{ucorr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Корреляции Pareto utilities")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_utility_correlation.png", dpi=180, bbox_inches="tight")
plt.show()

# Часть C. Global Pareto-front

## 14. Распределение глобальных Pareto ranks
**Pareto rank 0** — первый недоминируемый фронт. После его удаления строится следующий фронт и т.д. Rank — это не вероятность и не обычное место в рейтинге; он показывает уровень многокритериального доминирования.


In [ ]:
ranks = pd.to_numeric(all_unique["pareto_rank_global"], errors="coerce")
rank_counts = (
    ranks[ranks >= 0].astype(int).value_counts().sort_index()
    .rename_axis("pareto_rank_global").reset_index(name="count")
)
display(rank_counts.head(25))
rank_counts.to_csv(TABLE_DIR / "pareto_rank_counts.csv", index=False)

plt.figure(figsize=(10, 4.8))
plt.bar(rank_counts["pareto_rank_global"].astype(str), rank_counts["count"])
plt.xlabel("Global Pareto rank")
plt.ylabel("Количество")
plt.title("Распределение кандидатов по Pareto-front слоям")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(FIG_DIR / "07_pareto_rank_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

## 15. Какие свойства чаще являются bottleneck на global Pareto-front

Для каждого Pareto-кандидата берётся минимальная из семи utility. Свойство с минимальной utility считается текущим bottleneck этой молекулы.
**Идея bottleneck.** Для каждой молекулы берётся самая слабая utility. Если одно свойство часто оказывается минимальным, именно оно чаще всего ограничивает дальнейшее улучшение кандидатов.


In [ ]:
def add_bottleneck(df):
    out = df.copy()
    U = out[utility_cols].apply(pd.to_numeric, errors="coerce")
    out["bottleneck_property"] = U.idxmin(axis=1).str.replace("utility_", "", regex=False)
    out["bottleneck_utility"] = U.min(axis=1)
    return out

pareto_b = add_bottleneck(pareto)
counts = pareto_b["bottleneck_property"].value_counts().rename_axis("property").reset_index(name="count")
counts["label"] = counts["property"].map(lambda x: PROPS.get(x, {}).get("label", x))
display(counts)
counts.to_csv(TABLE_DIR / "pareto_bottleneck_counts.csv", index=False)

plt.figure(figsize=(10, 5.5))
plot_counts = counts.sort_values("count")
plt.barh(plot_counts["label"], plot_counts["count"])
plt.xlabel("Число Pareto-кандидатов")
plt.title("Чаще всего ограничивающее свойство на global Pareto-front")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_pareto_bottlenecks.png", dpi=180, bbox_inches="tight")
plt.show()

## 16. Интерактивные Pareto-графики

HTML-версии дополнительно сохраняются в папку `interactive` и попадут в итоговый ZIP анализа.
**Как использовать на защите.** Наводите курсор на точки и показывайте, что Pareto-front — не одна «лучшая» точка, а облако разных компромиссов. Две точки могут обе быть хорошими, если одна выигрывает по одному свойству, а вторая — по другому.


In [ ]:
import plotly.express as px

status_map = all_unique[["canonical_smiles"]].copy()
status_map["status"] = "Other"
feas = as_bool(all_unique["inside_both_ad"]) & as_bool(all_unique["sascore_pass"])
status_map.loc[feas, "status"] = "AD+SAS"
status_map.loc[pd.to_numeric(all_unique["pareto_rank_global"], errors="coerce").eq(0), "status"] = "Pareto"
status_map.loc[as_bool(all_unique["joint_pass"]), "status"] = "Strict"
status_map.loc[as_bool(all_unique["robust_joint_pass"]), "status"] = "Robust"

plot_df = pareto.copy().merge(status_map, on="canonical_smiles", how="left", suffixes=("", "_status"))

fig = px.scatter(
    plot_df,
    x="pred_group_A_absorption_max_nm",
    y="pred_group_A_log_half_life",
    color="status",
    size="quality_geomean",
    hover_name="canonical_smiles",
    hover_data=[
        "pred_group_A_log_extinction",
        "pred_group_A_photochem_efficiency",
        "pred_group_B_log_kp",
        "pred_group_B_skin_sensitization",
        "pred_group_B_skin_irritation",
        "deltaH_activation_proxy_kJ_mol",
    ],
    title="Global Pareto front: absorption vs half-life",
    labels={
        "pred_group_A_absorption_max_nm": "Absorption max, nm",
        "pred_group_A_log_half_life": "log10(t1/2, s)",
    },
)
fig.show()
fig.write_html(HTML_DIR / "pareto_absorption_vs_half_life.html")

parallel_df = pareto.copy()
fig2 = px.parallel_coordinates(
    parallel_df,
    dimensions=utility_cols,
    color="quality_geomean",
    range_color=[0, 1],
    title="Global Pareto front — 7 normalized utilities",
)
fig2.show()
fig2.write_html(HTML_DIR / "pareto_parallel_coordinates.html")

# Часть D. Почему 275 strict превращаются только в 3 robust

Robust-фильтр использует conservative margin:

- minimum objective: `prediction - threshold - 0.5 × uncertainty`;
- maximum objective: `threshold - prediction - 0.5 × uncertainty`;
- window objective: расстояние до ближайшей границы окна минус `0.5 × uncertainty`.

Чтобы кандидат был robust, **все семь conservative margins должны оставаться ≥ 0**.
**Смысл strict vs robust.** `strict` проверяет центральный прогноз. `robust` требует дополнительного запаса относительно порога с учётом uncertainty. Поэтому robust-shortlist намеренно маленький: он отсекает граничные решения, для которых небольшая ошибка модели меняет pass на fail.


In [ ]:
margin_cols = [f"margin_{k}" for k in PROPS]

strict_margin = strict.copy()
scaled_margin_cols = []
for key, cfg in PROPS.items():
    src = f"margin_{key}"
    dst = f"scaled_margin_{key}"
    strict_margin[dst] = pd.to_numeric(strict_margin[src], errors="coerce") / float(cfg["scale"])
    scaled_margin_cols.append(dst)

strict_margin["worst_scaled_margin"] = strict_margin[scaled_margin_cols].min(axis=1)
strict_margin["worst_margin_property"] = (
    strict_margin[scaled_margin_cols].idxmin(axis=1)
    .str.replace("scaled_margin_", "", regex=False)
)

cols_show = [
    "canonical_smiles", "quality_geomean", "worst_scaled_margin", "worst_margin_property", "robust_joint_pass"
]
display(strict_margin[cols_show].sort_values("worst_scaled_margin", ascending=False).head(20))
strict_margin[cols_show].to_csv(TABLE_DIR / "strict_robust_margin_analysis.csv", index=False)

plt.figure(figsize=(9, 5))
plt.hist(strict_margin["worst_scaled_margin"].dropna(), bins=40)
plt.axvline(0, linestyle="--", linewidth=1.5, label="Robust boundary")
plt.xlabel("Минимальный conservative margin / property scale")
plt.ylabel("Strict candidates")
plt.title("Почему большинство strict-кандидатов не проходят robust")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "09_strict_to_robust_margin.png", dpi=180, bbox_inches="tight")
plt.show()

print("Strict:", len(strict))
print("Robust:", len(robust))
print("Доля robust среди strict, %:", round(100 * len(robust) / max(len(strict), 1), 3))

## 18. Неопределённость evaluator-моделей

Для сопоставимости uncertainty делится на utility scale соответствующего свойства.
**Ограничение интерпретации.** Uncertainty здесь относится к конкретным surrogate evaluator-ам. Она не покрывает все источники неопределённости: domain shift, ошибки данных, несоответствие раствора и плёнки, фотопродукты, экспериментальные условия и т.д.


In [ ]:
unc_rows = []
for key, cfg in PROPS.items():
    col = f"unc_{key}"
    for subset_name, df in [("AD+SAS", feasible), ("Pareto", pareto), ("Strict", strict), ("Robust", robust)]:
        vals = pd.to_numeric(df[col], errors="coerce") / cfg["scale"]
        unc_rows.append([key, cfg["label"], subset_name, vals.mean(), vals.median(), vals.max()])

unc_summary = pd.DataFrame(unc_rows, columns=[
    "property", "label", "subset", "mean_unc_over_scale", "median_unc_over_scale", "max_unc_over_scale"
])
display(unc_summary.round(4))
unc_summary.to_csv(TABLE_DIR / "normalized_uncertainty_summary.csv", index=False)

# Часть E. AD, SAScore и novelty
**Три разных вопроса.** AD спрашивает «знакома ли модель с похожей химией?», SAScore — «насколько структура выглядит синтетически доступной по proxy?», novelty — «не является ли кандидат просто повторением известной структуры?». Высокий результат по одному из этих критериев не заменяет остальные.


In [ ]:
quality_summary = []
for name, df in [("Unique", all_unique), ("Pareto", pareto), ("Strict", strict), ("Robust", robust)]:
    novelty = 100 * as_bool(df["novelty"]).mean() if len(df) else np.nan
    ad_a = pd.to_numeric(df["ad_A"], errors="coerce")
    ad_b = pd.to_numeric(df["ad_B"], errors="coerce")
    sa = pd.to_numeric(df["sascore"], errors="coerce")
    quality_summary.append([
        name, len(df), novelty, ad_a.mean(), ad_b.mean(), sa.mean(),
        pd.to_numeric(df["quality_geomean"], errors="coerce").mean()
    ])

quality_summary = pd.DataFrame(quality_summary, columns=[
    "subset", "n", "novelty_%", "mean_ad_A", "mean_ad_B", "mean_SAScore", "mean_quality_geomean"
])
display(quality_summary.round(4))
quality_summary.to_csv(TABLE_DIR / "subset_quality_summary.csv", index=False)

plt.figure(figsize=(8.5, 4.8))
plt.hist(pd.to_numeric(all_unique["ad_A"], errors="coerce").dropna(), bins=35, alpha=0.55, label="AD A")
plt.hist(pd.to_numeric(all_unique["ad_B"], errors="coerce").dropna(), bins=35, alpha=0.55, label="AD B")
plt.axvline(AD_MAX, linestyle="--", linewidth=1.5, label=f"cutoff={AD_MAX}")
plt.xlabel("1 - max Tanimoto")
plt.ylabel("Unique candidates")
plt.title("Applicability Domain distances")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "10_ad_distances.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8.5, 4.8))
plt.hist(pd.to_numeric(all_unique["sascore"], errors="coerce").dropna(), bins=35)
plt.axvline(SAS_MAX, linestyle="--", linewidth=1.5, label=f"SAS cutoff={SAS_MAX}")
plt.xlabel("SAScore")
plt.ylabel("Unique candidates")
plt.title("Synthetic accessibility score")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "11_sascore_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

# Часть F. Химическое пространство — Morgan FP + PCA

PCA строится по тем же **1024-bit Morgan radius 2 fingerprints**, которые используются для AD. Это не часть обучения REINVENT; это только визуализация химического пространства итоговых кандидатов.
**Как читать PCA.** Это проекция высокоразмерных fingerprint-ов в две координаты для визуального обзора. Близость на PCA-графике полезна для общей картины, но не является точной метрикой химического сходства; для строгого сравнения лучше использовать исходный fingerprint/Tanimoto.


In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.decomposition import PCA

morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

valid_rows = []
fps_array = []
for idx, smi in all_unique["canonical_smiles"].items():
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    fp = morgan.GetFingerprint(mol)
    arr = np.zeros(1024, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    valid_rows.append(idx)
    fps_array.append(arr)

Xfp = np.asarray(fps_array, dtype=np.float32)
pca = PCA(n_components=2)
coords = pca.fit_transform(Xfp)

chem = all_unique.loc[valid_rows].copy().reset_index(drop=True)
chem["PC1"] = coords[:, 0]
chem["PC2"] = coords[:, 1]
chem["status"] = "Other"
mask_feas = as_bool(chem["inside_both_ad"]) & as_bool(chem["sascore_pass"])
chem.loc[mask_feas, "status"] = "AD+SAS"
chem.loc[pd.to_numeric(chem["pareto_rank_global"], errors="coerce").eq(0), "status"] = "Pareto"
chem.loc[as_bool(chem["joint_pass"]), "status"] = "Strict"
chem.loc[as_bool(chem["robust_joint_pass"]), "status"] = "Robust"

print("Explained variance PC1+PC2:", round(float(pca.explained_variance_ratio_[:2].sum()), 4))
chem.to_csv(TABLE_DIR / "chemical_space_pca.csv", index=False)

plt.figure(figsize=(9, 7))
for status in ["Other", "AD+SAS", "Pareto", "Strict", "Robust"]:
    d = chem[chem["status"] == status]
    if len(d):
        plt.scatter(d["PC1"], d["PC2"], s=22 if status != "Robust" else 90, alpha=0.7, label=f"{status} (n={len(d)})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Химическое пространство: Morgan FP → PCA")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "12_chemical_space_pca.png", dpi=180, bbox_inches="tight")
plt.show()

fig = px.scatter(
    chem,
    x="PC1", y="PC2", color="status",
    hover_name="canonical_smiles",
    hover_data=["quality_geomean", "pareto_rank_global", "source_profiles"],
    title="Chemical space PCA — интерактивная версия"
)
fig.show()
fig.write_html(HTML_DIR / "chemical_space_pca.html")

# Часть G. Scaffold-анализ

Murcko scaffolds помогают понять, найдено ли несколько разных химических семейств или Pareto-front концентрируется вокруг одного каркаса.
**Зачем scaffolds.** Большое число молекул ещё не означает химическое разнообразие. Если 300 кандидатов отличаются только небольшими заместителями вокруг одного ядра, пространство исследовано узко. Murcko scaffold показывает разнообразие основных каркасов.


In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold


def scaffold_from_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return None
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaf if scaf else "<acyclic>"
    except Exception:
        return None

scaffold_stats = []
scaffold_tables = {}
for name, df in [("Unique", all_unique), ("Pareto", pareto), ("Strict", strict), ("Robust", robust)]:
    s = df["canonical_smiles"].map(scaffold_from_smiles)
    n_unique = s.dropna().nunique()
    scaffold_stats.append([name, len(df), n_unique, len(df) / max(n_unique, 1)])
    scaffold_tables[name] = s.value_counts(dropna=True).rename_axis("scaffold").reset_index(name="count")

scaffold_summary = pd.DataFrame(scaffold_stats, columns=["subset", "molecules", "unique_scaffolds", "molecules_per_scaffold"])
display(scaffold_summary.round(3))
scaffold_summary.to_csv(TABLE_DIR / "scaffold_summary.csv", index=False)

pareto_scaf = scaffold_tables["Pareto"].head(20).sort_values("count")
display(scaffold_tables["Pareto"].head(20))
scaffold_tables["Pareto"].to_csv(TABLE_DIR / "pareto_scaffold_counts.csv", index=False)

plt.figure(figsize=(10, 7))
plt.barh(pareto_scaf["scaffold"], pareto_scaf["count"])
plt.xlabel("Количество молекул")
plt.title("Top-20 Murcko scaffolds на global Pareto-front")
plt.tight_layout()
plt.savefig(FIG_DIR / "13_pareto_scaffolds.png", dpi=180, bbox_inches="tight")
plt.show()

# Часть H. Детальный анализ strict и robust кандидатов

## 22. Удобная таблица для чтения
**Главный практический результат.** `strict` — широкий shortlist для дальнейшего ранжирования; `robust` — маленький приоритетный набор для более дорогих расчётов/проверок. Ни один из них пока нельзя называть экспериментально подтверждённым материалом.


In [ ]:
readable_cols = [
    "canonical_smiles",
    "source_profiles",
    "quality_geomean",
    "pareto_rank_global",
    "pred_group_A_absorption_max_nm",
    "pred_group_A_log_extinction",
    "pred_group_A_photochem_efficiency",
    "pred_group_A_log_half_life",
    "t_half_seconds_pred",
    "deltaH_activation_proxy_kJ_mol",
    "pred_group_B_log_kp",
    "pred_group_B_skin_sensitization",
    "pred_group_B_skin_irritation",
    "ad_A", "ad_B", "sascore", "novelty",
]
readable_cols = [c for c in readable_cols if c in all_unique.columns]


def readable(df):
    out = df[readable_cols].copy()
    if "t_half_seconds_pred" in out:
        out.insert(out.columns.get_loc("t_half_seconds_pred") + 1, "t_half_hours_pred", pd.to_numeric(out["t_half_seconds_pred"], errors="coerce") / 3600)
    return out

robust_readable = readable(robust).sort_values("quality_geomean", ascending=False)
strict_readable = readable(strict).sort_values("quality_geomean", ascending=False)
pareto_readable = readable(pareto).sort_values("quality_geomean", ascending=False)

print("ROBUST CANDIDATES:")
display(robust_readable.round(4))

print("TOP-20 STRICT по quality_geomean (это model-based screening score, не экспериментальный рейтинг):")
display(strict_readable.head(20).round(4))

robust_readable.to_csv(TABLE_DIR / "robust_candidates_readable.csv", index=False)
strict_readable.to_csv(TABLE_DIR / "strict_candidates_readable.csv", index=False)
pareto_readable.to_csv(TABLE_DIR / "global_pareto_front_readable.csv", index=False)

## 23. 2D-структуры robust-кандидатов

In [ ]:
from rdkit.Chem import Draw


def mol_grid(df, max_n=24, mols_per_row=3, sub_img_size=(360, 280)):
    subset = df.head(max_n).copy()
    mols, legends = [], []
    for i, row in subset.iterrows():
        mol = Chem.MolFromSmiles(str(row["canonical_smiles"]))
        if mol is None:
            continue
        q = pd.to_numeric(pd.Series([row.get("quality_geomean")]), errors="coerce").iloc[0]
        lam = pd.to_numeric(pd.Series([row.get("pred_group_A_absorption_max_nm")]), errors="coerce").iloc[0]
        logt = pd.to_numeric(pd.Series([row.get("pred_group_A_log_half_life")]), errors="coerce").iloc[0]
        dH = pd.to_numeric(pd.Series([row.get("deltaH_activation_proxy_kJ_mol")]), errors="coerce").iloc[0]
        legend = f"Q={q:.3f} | λ={lam:.0f} nm\nlog t1/2={logt:.2f} | ΔH‡≈{dH:.1f} kJ/mol"
        mols.append(mol)
        legends.append(legend)
    if not mols:
        print("Нет молекул для отображения")
        return None
    return Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=sub_img_size,
        legends=legends,
        useSVG=False,
    )

def save_grid_image(image_obj, path):
    path = Path(path)
    if hasattr(image_obj, "save"):
        image_obj.save(str(path))
        return
    data = getattr(image_obj, "data", None)
    if isinstance(data, bytes):
        path.write_bytes(data)
        return
    if isinstance(image_obj, bytes):
        path.write_bytes(image_obj)
        return
    raise TypeError(f"Не удалось сохранить RDKit grid типа {type(image_obj)}")

robust_grid = mol_grid(robust_readable, max_n=24, mols_per_row=3)
if robust_grid is not None:
    display(robust_grid)
    save_grid_image(robust_grid, FIG_DIR / "14_robust_molecule_grid.png")

## 24. 2D-структуры наиболее качественных Pareto-кандидатов

Сортировка здесь используется только для удобства просмотра по `quality_geomean`. Она **не заменяет Pareto-front** и не доказывает экспериментальное превосходство одной молекулы над другой.

In [ ]:
pareto_grid = mol_grid(pareto_readable, max_n=24, mols_per_row=4, sub_img_size=(320, 250))
if pareto_grid is not None:
    display(pareto_grid)
    save_grid_image(pareto_grid, FIG_DIR / "15_pareto_top24_molecule_grid.png")

# Часть I. Half-life и Eyring ΔH‡ proxy

`deltaH_activation_proxy_kJ_mol` вычисляется из предсказанного half-life. Поэтому это **не независимое восьмое свойство**, а монотонно связанный кинетический proxy.
**Критически важная оговорка.** Этот ΔH‡-proxy получен из predicted half-life через Eyring-приближение и допущение о малом вкладе ΔS‡. Он монотонно связан с half-life, поэтому не является независимой восьмой целью и не заменяет реальную энтальпию хранения между фотоизомерами.


In [ ]:
half_col = "pred_group_A_log_half_life"
dh_col = "deltaH_activation_proxy_kJ_mol"

plt.figure(figsize=(8, 5.5))
plt.scatter(
    pd.to_numeric(all_unique[half_col], errors="coerce"),
    pd.to_numeric(all_unique[dh_col], errors="coerce"),
    s=18, alpha=0.55
)
plt.xlabel("Predicted log10(t1/2, s)")
plt.ylabel("ΔH‡ activation proxy, kJ/mol")
plt.title("Eyring ΔH‡ proxy является функцией predicted half-life")
plt.tight_layout()
plt.savefig(FIG_DIR / "16_half_life_vs_deltaH_proxy.png", dpi=180, bbox_inches="tight")
plt.show()

if "t_half_seconds_pred" in all_unique.columns:
    t = pd.to_numeric(all_unique["t_half_seconds_pred"], errors="coerce") / 3600
    desc = t.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("t_half_hours_pred")
    display(desc)
    desc.to_csv(TABLE_DIR / "predicted_half_life_hours_summary.csv")

# Часть J. Происхождение кандидатов из семи агентов

Если одна структура появилась у нескольких профилей, это отражено в `source_profiles` через `;`.
**Почему это интересно.** Если одна и та же структура независимо появляется у нескольких priority-профилей, это означает, что разные направления оптимизации пришли к одному компромиссу. Это не доказательство физической истинности, но хороший сигнал устойчивости поискового результата внутри данной scoring-системы.


In [ ]:
def count_sources(x):
    if pd.isna(x) or not str(x).strip():
        return 0
    return len(set(str(x).split(";")))

all_unique["n_source_profiles"] = all_unique["source_profiles"].map(count_sources)
pareto_sources = all_unique[pd.to_numeric(all_unique["pareto_rank_global"], errors="coerce").eq(0)].copy()

source_count = pareto_sources["n_source_profiles"].value_counts().sort_index().rename_axis("n_profiles").reset_index(name="molecules")
display(source_count)
source_count.to_csv(TABLE_DIR / "pareto_source_profile_overlap.csv", index=False)

plt.figure(figsize=(8, 4.8))
plt.bar(source_count["n_profiles"].astype(str), source_count["molecules"])
plt.xlabel("В скольких Pareto-профилях встречалась молекула")
plt.ylabel("Global Pareto candidates")
plt.title("Пересечение результатов семи агентов")
plt.tight_layout()
plt.savefig(FIG_DIR / "17_source_profile_overlap.png", dpi=180, bbox_inches="tight")
plt.show()

if len(robust):
    robust_origins = robust[["canonical_smiles", "source_profiles"]].copy()
    robust_origins["n_source_profiles"] = robust_origins["source_profiles"].map(count_sources)
    display(robust_origins)

# Часть K. Автоматический итоговый отчёт в числах
**Назначение.** Эта ячейка собирает ключевые числа в одном месте, чтобы их можно было переносить в отчёт без ручного пересчёта. Для публикации/защиты значения всё равно следует связывать с точной версией архива и evaluator-ов.


In [ ]:
report = {
    "profiles": int(summary["profiles"]),
    "profile_rows_total": int(summary["profile_rows_total"]),
    "unique_candidates": int(summary["unique_candidates"]),
    "inside_both_ad_and_sas": int(summary["inside_both_ad_and_sas"]),
    "global_pareto_front": int(summary["global_pareto_front"]),
    "strict_candidates": int(summary["strict_candidates"]),
    "robust_candidates": int(summary["robust_candidates"]),
    "strict_fraction_of_unique_pct": 100 * len(strict) / max(len(all_unique), 1),
    "robust_fraction_of_unique_pct": 100 * len(robust) / max(len(all_unique), 1),
    "robust_fraction_of_strict_pct": 100 * len(robust) / max(len(strict), 1),
    "pareto_fraction_of_unique_pct": 100 * len(pareto) / max(len(all_unique), 1),
    "deltaH_note": summary.get("deltaH_note"),
    "oracles_used": summary.get("oracles_used"),
}

with open(OUT_DIR / "analysis_summary.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))

# Часть L. Экспорт результатов анализа

Будут сохранены:

- все созданные PNG-графики;
- интерактивные HTML-графики Plotly;
- CSV-таблицы анализа;
- `analysis_summary.json`;
- один архив `REINVENT4_MOST_PARETO_ANALYSIS.zip`.

In [ ]:
archive_base = BASE / "REINVENT4_MOST_PARETO_ANALYSIS"
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUT_DIR))

print("ГОТОВО")
print("Папка анализа:", OUT_DIR)
print("ZIP анализа:", zip_path)
print("Размер ZIP, MB:", round(zip_path.stat().st_size / 1024**2, 2))

print("\nСозданные таблицы:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p.name)

print("\nСозданные графики:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p.name)

print("\nИнтерактивные HTML:")
for p in sorted(HTML_DIR.glob("*.html")):
    print(" -", p.name)

## 29. Скачать ZIP с анализом

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Не Colab. Архив находится здесь:", zip_path)

# Как читать результаты

Основные файлы после выполнения notebook:

- `tables/robust_candidates_readable.csv` — **3 robust-кандидата** и их ключевые свойства;
- `tables/strict_candidates_readable.csv` — **275 strict-кандидатов**;
- `tables/global_pareto_front_readable.csv` — **319 Pareto-кандидатов**;
- `tables/property_pass_rates.csv` — какие свойства чаще всего ограничивают отбор;
- `tables/strict_robust_margin_analysis.csv` — почему 272 strict-кандидата не проходят robust;
- `tables/chemical_space_pca.csv` — координаты PCA химического пространства;
- `tables/pareto_scaffold_counts.csv` — химические scaffolds Pareto-front;
- `figures/14_robust_molecule_grid.png` — структуры robust-кандидатов;
- `interactive/chemical_space_pca.html` — интерактивная карта химического пространства;
- `interactive/pareto_parallel_coordinates.html` — интерактивное сравнение 7 Pareto utilities.

Для следующего научного этапа приоритетно смотреть **robust-кандидаты**, затем их независимую валидацию (oracle/DFT/TD-DFT/эксперимент). ML-отбор сам по себе не доказывает экспериментальную работоспособность или реальную `ΔH_storage`.
### Короткая формулировка для защиты

«Мы обучили семь копий REINVENT4 с одинаковым prior и семью разными priorities. Все молекулы оценивались одним набором из семи surrogate evaluator-ов. После объединения и дедупликации мы анализировали global Pareto front, затем strict- и robust-прохождение. Поэтому итог — не одна “лучшая” молекула, а набор компромиссных кандидатов с разной степенью вычислительной уверенности.»
